# Week 7: convolution, sampling and aliasing

**What we do.** We implement linear and circular convolution and check the convolution theorem, sample a band-limited signal and reconstruct it with Shannon's sinc formula, observe aliasing when sampling too slowly, and quantize a signal with a given number of bits.

**What to expect.** Convolution in time is multiplication in frequency. A signal sampled above twice its highest frequency can be recovered exactly from its samples; below that rate, high frequencies masquerade as low ones. Every extra bit of quantization improves the signal-to-noise ratio by about 6 dB.

**How to read the code.** All functions come from `wavelets_lab.sampling`. Time is measured in seconds and frequencies in hertz (oscillations per second).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from wavelets_lab import sampling

## Linear convolution: filtering a signal

**What we do.** We smooth a noisy signal by convolving it with a moving-average filter of length 9 and compare our convolution with NumPy's.

**What to expect.** The filtered signal is smoother (each output sample is the mean of 9 input samples) and 8 samples longer (`N + M - 1`). The difference from `np.convolve` is 0 up to rounding.

**How to read the code.** `sampling.convolve(x, h)` returns the full convolution. To compare signals of different lengths we plot the output against its own index; the filter delays the signal by `(M - 1) / 2 = 4` samples.

In [ ]:
rng = np.random.default_rng(0)
n = np.arange(100)
signal = np.sin(2 * np.pi * n / 50) + 0.3 * rng.standard_normal(n.size)
average = np.ones(9) / 9
filtered = sampling.convolve(signal, average)
plt.figure(figsize=(10, 3.5))
plt.plot(n, signal, label="noisy signal x")
plt.plot(np.arange(filtered.size) - 4, filtered, label="x * h (moving average of 9)")
plt.xlabel("sample index n")
plt.ylabel("value")
plt.legend()
plt.show()
np.abs(filtered - np.convolve(signal, average)).max()

## The convolution theorem

**What we do.** We compute a circular convolution directly and through the FFT.

**What to expect.** `DFT(x ⊛ h) = DFT(x) · DFT(h)`, so both results agree up to rounding (difference about `1e-15`). For long filters the FFT route is much faster.

**How to read the code.** `sampling.circular_convolve(x, h)` sums cyclically shifted copies of `x`; `np.fft.ifft(np.fft.fft(x) * np.fft.fft(h)).real` multiplies the spectra and transforms back.

In [ ]:
x, h = rng.standard_normal(64), rng.standard_normal(64)
direct = sampling.circular_convolve(x, h)
via_fft = np.fft.ifft(np.fft.fft(x) * np.fft.fft(h)).real
np.abs(direct - via_fft).max()

## Shannon reconstruction

**What we do.** We sample the signal `sin(2π·1.3 t) + 0.5 cos(2π·3.1 t)` (highest frequency 3.1 Hz) at 10 Hz for 10 seconds and reconstruct it between the samples with the sinc formula.

**What to expect.** 10 Hz is above the Nyquist rate `2 × 3.1 = 6.2 Hz`, so the reconstruction (dashed) lies on top of the true signal in the middle of the interval. Near the ends of the sampled interval small errors appear, because the infinite sum is truncated.

**How to read the code.** `sampling.sample(f, rate, duration)` needs the signal as a function of time; we pass a `lambda`, an anonymous one-line function `s ↦ value`. It returns the sampling instants and the values. On the fine grid `t` we evaluate the same formula directly (`exact`) and the reconstruction `sampling.sinc_interpolate(samples, rate, t)`, i.e. `Σ f(nT) sinc((t - nT)/T)`.

In [ ]:
rate = 10.0
t_samples, samples = sampling.sample(
    lambda s: np.sin(2 * np.pi * 1.3 * s) + 0.5 * np.cos(2 * np.pi * 3.1 * s),
    rate,
    10.0,
)
t = np.linspace(0.0, 10.0, 2000)
exact = np.sin(2 * np.pi * 1.3 * t) + 0.5 * np.cos(2 * np.pi * 3.1 * t)
reconstruction = sampling.sinc_interpolate(samples, rate, t)
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(t, exact, label="signal")
axes[0].plot(t, reconstruction, "--", label="sinc reconstruction")
axes[0].plot(t_samples, samples, "o", label="samples (10 Hz)")
axes[0].set_xlabel("time t [s]")
axes[0].set_ylabel("value")
axes[0].legend()
axes[1].plot(t, reconstruction - exact)
axes[1].set_xlabel("time t [s]")
axes[1].set_ylabel("reconstruction error")
plt.show()

## Aliasing

**What we do.** We sample a 9 Hz cosine at 10 Hz, i.e. below its Nyquist rate of 18 Hz.

**What to expect.** The samples lie exactly on a 1 Hz cosine: after sampling, 9 Hz and 1 Hz are indistinguishable (`|9 - 10| = 1`). This is why audio is low-pass filtered before sampling and why wheels in films seem to turn backwards.

**How to read the code.** `sampling.aliased_frequency(9, 10)` computes the apparent frequency (1 Hz). The fine curves show the true and the apparent signal, the dots the samples.

In [ ]:
rate, frequency = 10.0, 9.0
alias = sampling.aliased_frequency(frequency, rate)
t = np.linspace(0.0, 2.0, 2000)
t_samples = np.arange(0.0, 2.0, 1 / rate)
plt.figure(figsize=(10, 3.5))
plt.plot(
    t, np.cos(2 * np.pi * frequency * t), alpha=0.4, label=f"{frequency} Hz signal"
)
plt.plot(t, np.cos(2 * np.pi * alias * t), label=f"apparent {alias} Hz signal")
plt.plot(
    t_samples, np.cos(2 * np.pi * frequency * t_samples), "ko", label="samples at 10 Hz"
)
plt.xlabel("time t [s]")
plt.ylabel("value")
plt.legend(loc="lower right")
plt.show()

## Frequency folding

**What we do.** We plot the apparent frequency as a function of the true frequency for sampling at 10 Hz.

**What to expect.** A zigzag ("folding") line: frequencies up to the Nyquist frequency 5 Hz are seen correctly, frequencies between 5 and 10 Hz fold back down to 0, then the pattern repeats.

**How to read the code.** A list comprehension calls `sampling.aliased_frequency` for each true frequency.

In [ ]:
true_frequencies = np.linspace(0.0, 25.0, 501)
apparent = [sampling.aliased_frequency(f, 10.0) for f in true_frequencies]
plt.figure(figsize=(8, 3.5))
plt.plot(true_frequencies, apparent)
plt.axhline(5.0, color="k", linestyle="--", label="Nyquist frequency 5 Hz")
plt.xlabel("true frequency [Hz]")
plt.ylabel("apparent frequency [Hz]")
plt.legend()
plt.show()

## Quantization: about 6 dB per bit

**What we do.** We quantize a sine wave with 3 bits (8 levels) to see the staircase, and then with 1 to 12 bits to measure the signal-to-quantization-noise ratio (SQNR).

**What to expect.** The 3-bit signal is a staircase with 8 steps. The SQNR grows linearly with the number of bits, with slope about 6.02 dB per bit (each bit halves the step and so the error amplitude). CD audio uses 16 bits, i.e. about 98 dB.

**How to read the code.** `sampling.quantize(x, n_bits)` rounds to the centres of `2^n_bits` cells on `[-1, 1]`; `sampling.sqnr(x, q)` computes `10 log10(P_signal / P_error)`. The dashed line is the rule of thumb `6.02 b + 1.76` dB for a full-scale sine.

In [ ]:
x = np.sin(np.linspace(0.0, 2 * np.pi, 10000))
bits = np.arange(1, 13)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(x[:2500], label="signal")
axes[0].plot(sampling.quantize(x, 3)[:2500], label="3 bits")
axes[0].set_xlabel("sample index")
axes[0].set_ylabel("value")
axes[0].legend()
axes[1].plot(
    bits,
    [sampling.sqnr(x, sampling.quantize(x, b)) for b in bits],
    "o-",
    label="measured",
)
axes[1].plot(bits, 6.02 * bits + 1.76, "k--", label="6.02 b + 1.76")
axes[1].set_xlabel("bits b")
axes[1].set_ylabel("SQNR [dB]")
axes[1].legend()
plt.tight_layout()
plt.show()

## Reference: scripts and functions used in this notebook

**What we do.** We list every function of the course package used above with its purpose, input, output and a recipe for writing it yourself.

**What to expect.** One module is used: `wavelets_lab.sampling` (`src/wavelets_lab/sampling.py`).

**How to read the code.** Signals are 1D arrays; `rate` is the number of samples per second.

**Library functions called directly in this notebook.**

| Function | What it does | Input → output |
|---|---|---|
| `np.convolve(x, h)` | NumPy's linear convolution (reference) | arrays → array of length `N + M - 1` |
| `np.fft.fft`, `np.fft.ifft` | DFT and inverse DFT | array → complex array |
| `lambda t: ...` | defines a short anonymous function | expression → function |
| `plt.axhline(y)` | horizontal line at height `y` | number → line |
| `plt.subplots(..., sharex=True)` | panels with a common horizontal axis | – |

### Module `wavelets_lab.sampling` (file `src/wavelets_lab/sampling.py`)

Discrete convolution, sampling of functions, Shannon (sinc) reconstruction, aliasing and quantization of 1D signals. Only NumPy is imported.

#### `sampling.convolve(x, h)`

**What it is.** The full linear convolution `y[n] = sum_k h[k] x[n - k]` of two finite signals, equal to `numpy.convolve`. Filtering a signal with a filter `h` is exactly this operation.

**Input.** `x` (array-like, length `N`): signal; `h` (array-like, length `M`): filter.

**Output.** `float64` array of length `N + M - 1`.

**How to write it yourself.**
1. Convert both to `float64`; `result = np.zeros(N + M - 1)`.
2. Every tap `h[k]` contributes the signal shifted by `k` and multiplied by `h[k]`: for `shift, tap in enumerate(h)`: `result[shift : shift + N] += tap * x`.
3. Return `result`. (This loops over the `M` taps, not over the `N` samples, so it is fast for short filters.)

#### `sampling.circular_convolve(x, h)`

**What it is.** The circular (periodic) convolution `y[n] = sum_k h[k] x[(n - k) mod N]` of two signals of equal length. By the convolution theorem, `DFT(y) = DFT(x) · DFT(h)`.

**Input.** `x`, `h` (array-like, same length `N`).

**Output.** `float64` array of length `N`. Raises `ValueError` for different lengths.

**How to write it yourself.**
1. Convert to `float64` and check equal shapes.
2. `np.roll(x, k)` shifts the signal cyclically by `k`: entry `n` becomes `x[(n - k) mod N]`.
3. Return the sum over all taps: `sum(tap * np.roll(x, k) for k, tap in enumerate(h))`.

#### `sampling.sample(f, rate, duration)`

**What it is.** Uniform sampling of a function of time on `[0, duration)` with `rate` samples per unit of time (sampling period `T = 1 / rate`).

**Input.**
- `f` (callable): vectorized function of time.
- `rate` (`float`): sampling rate, e.g. in Hz.
- `duration` (`float`): length of the interval.

**Output.** Tuple `(t, samples)`: the instants `t[n] = n / rate` and the values `f(t[n])`.

**How to write it yourself.**
1. Number of samples: `int(np.floor(duration * rate))`.
2. `t = np.arange(count) / rate`.
3. Return `t, np.asarray(f(t), dtype=np.float64)`.

#### `sampling.sinc_interpolate(samples, rate, t)`

**What it is.** Shannon's reconstruction formula `f(t) = sum_n f(nT) sinc((t - nT) / T)` with `T = 1 / rate` and `sinc(u) = sin(πu) / (πu)`. By the sampling theorem it recovers a band-limited signal exactly if the sampling rate is above twice its highest frequency (the Nyquist rate); with finitely many samples the reconstruction is accurate away from the ends of the sampled interval.

**Input.**
- `samples` (array-like): `f(nT)` for `n = 0, 1, …`.
- `rate` (`float`): sampling rate `1 / T`.
- `t` (array-like): points where the reconstruction is evaluated.

**Output.** `float64` array of reconstructed values.

**How to write it yourself.**
1. Sampling instants: `nodes = np.arange(len(samples)) / rate`.
2. Matrix of shifted sincs, one row per evaluation point: `S = np.sinc(rate * np.subtract.outer(t, nodes))` (NumPy's `np.sinc` is already the normalized `sin(πu)/(πu)`).
3. Return `S @ samples`.

#### `sampling.aliased_frequency(frequency, rate)`

**What it is.** The apparent frequency of a sinusoid of frequency `f` sampled at rate `fs`: the samples are identical to those of a sinusoid of frequency `|f - k fs|` for every integer `k`, and the one in `[0, fs / 2]` is what we "see". Frequencies above the Nyquist frequency `fs / 2` fold back (aliasing).

**Input.** `frequency` (`float`), `rate` (`float`).

**Output.** `float` in `[0, rate / 2]`.

**How to write it yourself.**
1. Reduce modulo the rate: `folded = abs(frequency) % rate`.
2. Fold the upper half back: return `min(folded, rate - folded)`.

#### `sampling.quantize(x, n_bits, value_range=(-1.0, 1.0))`

**What it is.** Uniform quantization with `2 ** n_bits` levels: the interval `value_range` is cut into equal cells and every value is replaced by the centre of its cell (values outside are clipped to the first or last cell). This is what an analog-to-digital converter does.

**Input.**
- `x` (array-like): signal.
- `n_bits` (`int`, at least 1): bits per sample.
- `value_range` (`tuple[float, float]`): `(low, high)` with `low < high`.

**Output.** `float64` array of quantized values. Raises `ValueError` for `n_bits < 1` or an empty range.

**How to write it yourself.**
1. Check the arguments.
2. `levels = 2 ** n_bits`, cell width `cell = (high - low) / levels`.
3. Cell index: `index = np.floor((x - low) / cell)`, clipped to `[0, levels - 1]`.
4. Return the cell centres `low + (index + 0.5) * cell`.

#### `sampling.sqnr(signal, quantized)`

**What it is.** Signal-to-quantization-noise ratio in dB, `10 log10(P_signal / P_noise)` where `P` is the mean square and the noise is `signal - quantized`. Theory says each extra bit adds about 6.02 dB.

**Input.** `signal`, `quantized` (array-like, same length).

**Output.** `float` (dB), `inf` if there is no error.

**How to write it yourself.**
1. `noise = signal - quantized`; `noise_power = np.mean(noise**2)`.
2. If `noise_power == 0` return `float("inf")`.
3. Return `10 * np.log10(np.mean(signal**2) / noise_power)`.